clone the repo if you are on cloud

In [ ]:
# !git clone --depth 1 https://github.com/OussamaMadev/face-anti-spoofing-pipline
# %cd face-anti-spoofing-pipline

In [1]:
import numpy as np
import tensorflow as tf
import trainingPipeline
from trainingPipeline import TrainingPipeline
import architectures
from tensorflow.keras import layers, models

you can define multiple keras models here 
and file the configes list then run multiple training at ones

check models.py

be carefull with paths

double check you configs befor launching long experiments

In [2]:

def init_model(input_shape=(224, 224, 3)):
    input_rgb = layers.Input(shape=input_shape, name="input_rgb")
    
    # 1. Color Space Transformations
    input_hsv = layers.Lambda(lambda x: tf.image.rgb_to_hsv(x), name="rgb_to_hsv")(input_rgb)
    input_ycbcr = layers.Lambda(lambda x: tf.image.rgb_to_yuv(x), name="rgb_to_ycbcr")(input_rgb)
    
    # 2. Multi-Channel Fusion with initial Noise Filtering
    merged = layers.Concatenate(axis=-1)([input_rgb, input_hsv, input_ycbcr])
    
    # 3. Entry Block: Depthwise Separable to treat color channels independently first
    x = layers.SeparableConv2D(64, (3, 3), padding='same', activation='relu')(merged)
    x = layers.BatchNormalization()(x)
    
    # 4. Residual Blocks (Better for deeper learning without vanishing gradients)
    def res_block(tensor, filters):
        shortcut = layers.Conv2D(filters, (1, 1), padding='same')(tensor)
        
        val = layers.Conv2D(filters, (3, 3), padding='same', activation='relu')(tensor)
        val = layers.BatchNormalization()(val)
        val = layers.Conv2D(filters, (3, 3), padding='same')(val)
        val = layers.BatchNormalization()(val)
        
        return layers.Add()([shortcut, val])

    x = res_block(x, 64)
    x = layers.MaxPooling2D((2, 2))(x) # 112x112
    
    x = res_block(x, 128)
    # Simple Attention Mechanism: Focus on face texture, ignore background
    attention = layers.Conv2D(1, (1, 1), activation='sigmoid')(x)
    x = layers.Multiply()([x, attention])
    x = layers.MaxPooling2D((2, 2))(x) # 56x56
    
    # 5. Spatial Pyramid Pooling (SPP) - Captures multi-scale spoofing cues
    # Useful for finding tiny screen pixels vs. large paper edges
    pool1 = layers.GlobalAveragePooling2D()(x)
    pool2 = layers.GlobalMaxPooling2D()(x)
    
    x = layers.Concatenate()([pool1, pool2])
    
    # 6. Dense Neck with stronger Regularization
    x = layers.Dense(256, activation='relu', kernel_regularizer='l2')(x)
    x = layers.Dropout(0.6)(x) # Higher dropout to prevent overfitting
    x = layers.Dense(64, activation='relu')(x)
    
    output = layers.Dense(1, activation='sigmoid', name="classifier")(x)
    
    return models.Model(inputs=input_rgb, outputs=output, name = "MC-ResNet")



print("Model name: " + init_model().name)
print("Number of parameters: " + str(init_model().count_params()))


Model name: MC-ResNet
Number of parameters: 392723


In [ ]:
configs_example = [
    {
        "filtering_params": {
            "data_map_path": "data_maps/CASIA_FASD_v3_all.json",
        },
        
        "data_params": {
            "dataset_path": "data/CASIA_FASD_v3_224x224/DATA",
            "input_size": [224,224,3],
            "pixel_range": [0.0,1.0],
            "batch_size": 32,
            "validation_dataset_baleance": 1, # 0 for unbalanced, 1 for balanced 
        },
        
        "augmentation_params": {
            "rotation_range": 0.15,
            "horizontal_flip": 0,
            "brightness_range": [0.8,1.2],
            "zoom_range": 0.1
        },
        
        "model_params": {
            "model_init_function": init_model,  # Reference to the model initialization function
            "architecture": None,  # To be filled after model initialization
            "parameters_count": None,  # To be filled after model initialization
            "note" : None # optional note about the model architecture or the weights used (if any)
                       
        },
        
        "training_params": {
            "early_stopping_patience": 5,
            "learning_rate": 0.0001,
            "steps_per_epoch": 10,
            "initial_epochs": 1
        }      
    }
]


In [5]:
trainPipe = TrainingPipeline(configs_example, 
                             experiment_id="experiment_001" ,
                             logs_output_path="./experiment_records", 
                             models_output_path="./experiment_records", 
                             note="Testing the training pipeline with a simple configuration.")



All configurations validated successfully.


In [ ]:
trainPipe.run()

Starting Suite: experiment_001_20260414-184405

--- Running Sub-Experiment 1/1 ---
Configuration:
{'filtering_params': {'data_map_path': 'data_maps/CASIA_FASD_v3_all.json', 'keep_ratio': 1.0, 'filter_function': 'uniform_sampling'}, 'data_params': {'dataset_path': 'data/CASIA_FASD_v3_224x224/DATA', 'input_size': [224, 224, 3], 'pixel_range': [0.0, 1.0], 'batch_size': 32, 'validation_dataset_baleance': 0}, 'augmentation_params': {'rotation_range': 0.15, 'horizontal_flip': 0, 'brightness_range': [0.8, 1.2], 'zoom_range': 0.1}, 'model_params': {'model_init_function': <function init_model at 0x000001B2FCF696C0>, 'architecture': 'MC-ResNet', 'parameters_count': 392723, 'note': None}, 'training_params': {'early_stopping_patience': 5, 'learning_rate': 0.0001, 'steps_per_epoch': 10, 'initial_epochs': 1}}
Successfully loaded 50 subjects from JSON.
is_val_ds_balanced: False


: 